: 

loaded burgers.mat: t (201,) x (512,) usol (201, 512) nu 0.003183098861837907
Domain: x in [-1.0, 1.0], t in [0.0, 1.0]
Traditional ALM-PINN with L-BFGS inner solve:
  fixed points
  no jitter
  no resampling
  theta inner optimizer: scipy L-BFGS-B
  lambda update: lambda <- lambda + rho*c
  rho update: blockwise if feasibility stalls
BC_MODE = zero
USE_PDE_ANCHORS = True
n_params = 1981
constraints: pde 900 bc 200 ic 200 total 1300
objective points = 6400
network = [2, 30, 30, 30, 1]
W_PDE_OBJ = 1.0
TOTAL_LBFGS_BUDGET = 50000
MAX_OUTER = 100 INNER_MAXITER = 500
rho0: pde 1.0 bc 1.0 ic 10.0

[OUTER 1/100] rho_pde=1.000e+00, rho_bc=1.000e+00, rho_ic=1.000e+01
[ALM] outer=1 loss=3.015e-02 obj=1.481e-02 pde=(1.153e-01,7.969e-01) bc=(4.268e-03,9.497e-03) ic=(4.169e-02,1.460e-01) relL2=1.512e-01 lbfgs_success=False nit=500 msg='STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT'
      lam_rms: pde=1.153e-01, bc=4.268e-03, ic=4.169e-01
      >>> new best relL2=1.512e-01; saved -> theta_traditional_

In [0]:

# ============================================================
# TRADITIONAL ALM-PINN FOR VISCOUS_ BURGERS (JAX + SciPy L-BFGS)
#
# This is NOT AL-PINN gradient-ascent-on-lambda.
# This is classical augmented Lagrangian:
#
#   theta^{k+1} ≈ argmin_theta L_rho(theta, lambda^k)
#   lambda^{k+1} = lambda^k + rho_k c(theta^{k+1})
#
# Inner theta solver:
#   deterministic full-batch L-BFGS-B from scipy.optimize.minimize
#
# Points:
#   fixed once, no jitter, no resampling
#
# Default point counts, matching your SQP-PINN Burgers style:
#   PDE objective:   80 x 80 = 6400
#   PDE anchors:     30 x 30 = 900     optional hard constraints
#   BC:              100 time points
#   IC:              200 x points
#
# Default network:
#   [2] + [30]*3 + [1]
#
# Burgers:
#   u_t + u u_x - nu u_xx = 0
#
# Default BC:
#   zero Dirichlet: u(-1,t)=0, u(1,t)=0
#
# To match your SQP code's periodic constraints instead:
#   set BC_MODE = "periodic_value_deriv"
#
# This version is configured for a total L-BFGS-B budget of 20000.
# Recommended first run:
#   USE_PDE_ANCHORS = True
#   BC_MODE = "zero"
#   rho_pde0 = 1
#   rho_bc0  = 1
#   rho_ic0  = 10
#   inner_maxiter = 100
#   max_outer = 20
# ============================================================

import os
import time
import math
from functools import partial

import numpy as np
import scipy.io
import scipy.optimize

# -----------------------------
# Device
# -----------------------------
USE_CPU = False
CUDA_DEVICE = "0"

if USE_CPU:
    os.environ["JAX_PLATFORMS"] = "cpu"
else:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", CUDA_DEVICE)

os.environ.setdefault("XLA_FLAGS", "--xla_gpu_autotune_level=0")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import jax
jax.config.update("jax_default_matmul_precision", "tensorfloat32")

import jax.numpy as jnp
from jax import random, grad, vmap, hessian


# ============================================================
# Precision
# ============================================================
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)


# ============================================================
# User settings
# ============================================================
DATA_PATH = "data/burgers.mat"

# If True, ALM constraints include PDE anchor residuals, matching your SQP style.
# If False, ALM constraints are only BC+IC, closer to standard ALM-PINN.
USE_PDE_ANCHORS = True

# Options:
#   "zero"                 -> u(-1,t)=0 and u(1,t)=0
#   "periodic_value"       -> u(-1,t)-u(1,t)=0
#   "periodic_value_deriv" -> u(-1,t)-u(1,t)=0 and ux(-1,t)-ux(1,t)=0
BC_MODE = "zero"

# Point counts
NX_PDE_OBJ = 80
NT_PDE_OBJ = 80
NX_PDE_CON = 30
NT_PDE_CON = 30
K_BC = 100
K_IC = 200

# Network
HIDDEN_DIM = 30
NUM_HIDDEN = 3

# Objective weight. Use 10 to match your SQP Burgers setup.
W_PDE_OBJ = 1.0

# ALM outer/inner
# "Total 20000" means MAX_OUTER * INNER_MAXITER = 20000 L-BFGS-B iterations.
# Do NOT set MAX_OUTER=20000. That would be enormous.
TOTAL_LBFGS_BUDGET = 20000
INNER_MAXITER = 100       # L-BFGS-B iterations per ALM outer loop
MAX_OUTER = int(math.ceil(TOTAL_LBFGS_BUDGET / INNER_MAXITER))
PRINT_INNER = False       # scipy does not print inner by default

# L-BFGS options
LBFGS_FTOL = 1e-12
LBFGS_GTOL = 1e-8
LBFGS_MAXLS = 50
LBFGS_MAXCOR = 50

# Initial rho values
RHO_PDE0 = 1.0
RHO_BC0  = 1.0
RHO_IC0  = 10.0

# Rho update
# Gentler than the previous run. The previous 2.0/0.7 rule drove rho to 1e5 too quickly,
# which makes the ALM subproblem stiff and causes oscillatory output.
RHO_GROWTH = 1.5
RHO_MAX = 1e4
IMPROVE_THRESHOLD = 0.9   # if rms_new > 0.9*rms_old, increase rho

# Save
SAVE_NAME = "theta_traditional_alm_burgers_lbfgs_20000_best.npy"

# Seed
SEED = 0


# ============================================================
# Sizes
# ============================================================
K_PDE_OBJ = NX_PDE_OBJ * NT_PDE_OBJ
K_PDE_CON = NX_PDE_CON * NT_PDE_CON


def get_constraint_sizes(use_pde_anchors=USE_PDE_ANCHORS, bc_mode=BC_MODE):
    n_pde = K_PDE_CON if use_pde_anchors else 0

    if bc_mode == "zero":
        n_bc = 2 * K_BC
    elif bc_mode == "periodic_value":
        n_bc = K_BC
    elif bc_mode == "periodic_value_deriv":
        n_bc = 2 * K_BC
    else:
        raise ValueError(f"Unknown BC_MODE={bc_mode}")

    n_ic = K_IC
    return n_pde, n_bc, n_ic, n_pde + n_bc + n_ic


# ============================================================
# Network
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)

    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})

    return params


def mlp_apply(params, X):
    # Input order: [x, t], same as your SQP code.
    h = X

    for i, layer in enumerate(params):
        h = h @ layer["W"] + layer["b"]
        if i < len(params) - 1:
            h = jnp.tanh(h)

    return h


def flatten_params(params):
    flat_parts = []
    shapes = []

    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))

    return jnp.concatenate(flat_parts).astype(DTYPE), tuple(shapes)


def unflatten_params(theta, shapes):
    params = []
    idx = 0

    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)

        W = theta[idx:idx + W_size].reshape(W_shape)
        idx += W_size

        b = theta[idx:idx + b_size].reshape(b_shape)
        idx += b_size

        params.append({"W": W, "b": b})

    if idx != theta.size:
        raise ValueError(f"Used {idx} parameters but theta has size {theta.size}")

    return params


# ============================================================
# Data
# ============================================================
def load_burgers_mat(path):
    d = scipy.io.loadmat(path)
    t = d["t"].squeeze()
    x = d["x"].squeeze()
    usol = d["usol"]
    nu = float(np.array(d["nu"]).squeeze())
    return t, x, usol, nu


def make_grid_points(x_np, t_np):
    T, X = np.meshgrid(t_np, x_np, indexing="ij")
    X_grid = np.stack([X.reshape(-1), T.reshape(-1)], axis=1)
    return X_grid, T.shape


# ============================================================
# Fixed point sets
# ============================================================
def sample_pde_obj_fixed(key, x_min, x_max, t_min, t_max):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(NX_PDE_OBJ)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(NT_PDE_OBJ)

    it, ix = jnp.meshgrid(
        jnp.arange(NT_PDE_OBJ),
        jnp.arange(NX_PDE_OBJ),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K_PDE_OBJ, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt

    return jnp.stack([xs, ts], axis=1)


def sample_pde_con_fixed(key, x_min, x_max, t_min, t_max):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(NX_PDE_CON)
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(NT_PDE_CON)

    it, ix = jnp.meshgrid(
        jnp.arange(NT_PDE_CON),
        jnp.arange(NX_PDE_CON),
        indexing="ij",
    )

    it = it.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)

    x0 = DTYPE(x_min) + DTYPE(ix) * dx
    t0 = DTYPE(t_min) + DTYPE(it) * dt

    u = random.uniform(key, (K_PDE_CON, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = x0 + u[:, 0] * dx
    ts = t0 + u[:, 1] * dt

    return jnp.stack([xs, ts], axis=1)


def sample_bc_fixed(key, x_min, x_max, t_min, t_max):
    dt = (DTYPE(t_max) - DTYPE(t_min)) / DTYPE(K_BC)

    j = jnp.arange(K_BC, dtype=jnp.int32)
    t0 = DTYPE(t_min) + DTYPE(j) * dt

    u = random.uniform(key, (K_BC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (t0 + u * dt).reshape(-1, 1)

    XL = jnp.concatenate([DTYPE(x_min) * jnp.ones_like(ts), ts], axis=1)
    XR = jnp.concatenate([DTYPE(x_max) * jnp.ones_like(ts), ts], axis=1)

    return XL, XR


def sample_ic_fixed(key, x_min, x_max, t0, x_grid, u0_grid):
    dx = (DTYPE(x_max) - DTYPE(x_min)) / DTYPE(K_IC)

    j = jnp.arange(K_IC, dtype=jnp.int32)
    x0 = DTYPE(x_min) + DTYPE(j) * dx

    u = random.uniform(key, (K_IC,), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (x0 + u * dx).reshape(-1, 1)

    ts = DTYPE(t0) * jnp.ones_like(xs)
    Xic = jnp.concatenate([xs, ts], axis=1)

    u0 = jnp.interp(xs[:, 0], x_grid, u0_grid)

    return Xic, u0


# ============================================================
# PDE residual and BC derivatives
# ============================================================
def pde_residual(params, X, nu):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    H = vmap(hessian(u_fun))(X)

    u_x = du[:, 0]
    u_t = du[:, 1]
    u_xx = H[:, 0, 0]

    return u_t + u * u_x - DTYPE(nu) * u_xx


def u_and_ux(params, X):
    def u_fun(xt):
        return mlp_apply(params, xt[None, :])[0, 0]

    u = vmap(u_fun)(X)
    du = vmap(grad(u_fun))(X)
    ux = du[:, 0]
    return u, ux


# ============================================================
# Objective and constraints
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def pde_objective(theta, shapes, X_obj, nu, w_pde_obj):
    params = unflatten_params(theta, shapes)
    r = pde_residual(params, X_obj, nu)
    return DTYPE(w_pde_obj) * jnp.mean(r ** 2)


@partial(jax.jit, static_argnames=("shapes", "use_pde_anchors", "bc_mode"))
def constraint_blocks(theta, shapes,
                      X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic, nu,
                      use_pde_anchors: bool,
                      bc_mode: str):
    params = unflatten_params(theta, shapes)

    if use_pde_anchors:
        c_pde = pde_residual(params, X_pde_con, nu)
    else:
        c_pde = jnp.zeros((0,), dtype=DTYPE)

    if bc_mode == "zero":
        uL = mlp_apply(params, X_bc_L)[:, 0]
        uR = mlp_apply(params, X_bc_R)[:, 0]
        c_bc = jnp.concatenate([uL, uR], axis=0)

    elif bc_mode == "periodic_value":
        uL = mlp_apply(params, X_bc_L)[:, 0]
        uR = mlp_apply(params, X_bc_R)[:, 0]
        c_bc = uL - uR

    elif bc_mode == "periodic_value_deriv":
        uL, uxL = u_and_ux(params, X_bc_L)
        uR, uxR = u_and_ux(params, X_bc_R)
        c_bc = jnp.concatenate([uL - uR, uxL - uxR], axis=0)

    else:
        raise ValueError(f"Unknown bc_mode={bc_mode}")

    u_ic = mlp_apply(params, X_ic)[:, 0]
    c_ic = u_ic - u0_ic

    return c_pde, c_bc, c_ic


@partial(jax.jit, static_argnames=("shapes", "use_pde_anchors", "bc_mode"))
def full_constraint_vector(theta, shapes,
                           X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic, nu,
                           use_pde_anchors: bool,
                           bc_mode: str):
    c_pde, c_bc, c_ic = constraint_blocks(
        theta, shapes,
        X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic, nu,
        use_pde_anchors, bc_mode,
    )
    return jnp.concatenate([c_pde, c_bc, c_ic], axis=0)


@partial(jax.jit, static_argnames=("shapes", "use_pde_anchors", "bc_mode"))
def alm_loss_and_grad(theta, shapes,
                      lam_pde, lam_bc, lam_ic,
                      rho_pde, rho_bc, rho_ic,
                      X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
                      nu, w_pde_obj,
                      use_pde_anchors: bool,
                      bc_mode: str):
    def loss_fn(th):
        obj = pde_objective(th, shapes, X_obj, nu, w_pde_obj)

        c_pde, c_bc, c_ic = constraint_blocks(
            th, shapes,
            X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic, nu,
            use_pde_anchors, bc_mode,
        )

        # If c_pde is empty, this returns zero.
        if use_pde_anchors:
            alm_pde = jnp.mean(lam_pde * c_pde) + DTYPE(0.5) * DTYPE(rho_pde) * jnp.mean(c_pde ** 2)
            pde_rms = jnp.sqrt(jnp.mean(c_pde ** 2) + EPS)
            pde_max = jnp.max(jnp.abs(c_pde))
        else:
            alm_pde = DTYPE(0.0)
            pde_rms = DTYPE(0.0)
            pde_max = DTYPE(0.0)

        alm_bc = jnp.mean(lam_bc * c_bc) + DTYPE(0.5) * DTYPE(rho_bc) * jnp.mean(c_bc ** 2)
        alm_ic = jnp.mean(lam_ic * c_ic) + DTYPE(0.5) * DTYPE(rho_ic) * jnp.mean(c_ic ** 2)

        loss = obj + alm_pde + alm_bc + alm_ic

        aux = {
            "obj": obj,
            "alm_pde": alm_pde,
            "alm_bc": alm_bc,
            "alm_ic": alm_ic,
            "pde_rms": pde_rms,
            "bc_rms": jnp.sqrt(jnp.mean(c_bc ** 2) + EPS),
            "ic_rms": jnp.sqrt(jnp.mean(c_ic ** 2) + EPS),
            "pde_max": pde_max,
            "bc_max": jnp.max(jnp.abs(c_bc)),
            "ic_max": jnp.max(jnp.abs(c_ic)),
        }
        return loss, aux

    (loss, aux), g = jax.value_and_grad(loss_fn, has_aux=True)(theta)
    return loss, g, aux


# ============================================================
# Evaluation
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def predict_u(theta, shapes, X):
    params = unflatten_params(theta, shapes)
    return mlp_apply(params, X)[:, 0]


def eval_full_grid(theta, shapes, x_np, t_np, usol_np):
    X_grid_np, grid_shape = make_grid_points(x_np, t_np)
    X_grid = jnp.array(X_grid_np, dtype=DTYPE)

    u_pred = np.array(predict_u(theta, shapes, X_grid)).reshape(grid_shape)
    u_true = np.asarray(usol_np, dtype=np.float64)

    mse = float(np.mean((u_pred - u_true) ** 2))
    rel_l2 = float(np.linalg.norm(u_pred - u_true) / (np.linalg.norm(u_true) + 1e-12))

    return mse, rel_l2, u_pred


def block_stats(arr):
    arr = np.asarray(arr)
    if arr.size == 0:
        return 0.0, 0.0
    return float(np.sqrt(np.mean(arr ** 2) + 1e-30)), float(np.max(np.abs(arr)))


# ============================================================
# SciPy L-BFGS inner solve
# ============================================================
def lbfgs_inner_solve(theta, shapes,
                      lam_pde, lam_bc, lam_ic,
                      rho_pde, rho_bc, rho_ic,
                      X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
                      nu, w_pde_obj,
                      use_pde_anchors, bc_mode,
                      maxiter):
    def fun_and_jac(theta_np):
        th = jnp.asarray(theta_np, dtype=DTYPE)
        loss, g, _ = alm_loss_and_grad(
            th, shapes,
            lam_pde, lam_bc, lam_ic,
            DTYPE(rho_pde), DTYPE(rho_bc), DTYPE(rho_ic),
            X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
            DTYPE(nu), DTYPE(w_pde_obj),
            use_pde_anchors, bc_mode,
        )
        return float(loss), np.asarray(g, dtype=np.float64)

    theta0_np = np.asarray(theta, dtype=np.float64)

    res = scipy.optimize.minimize(
        fun_and_jac,
        theta0_np,
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": int(maxiter),
            "ftol": LBFGS_FTOL,
            "gtol": LBFGS_GTOL,
            "maxls": LBFGS_MAXLS,
            "maxcor": LBFGS_MAXCOR,
            "disp": False,
        },
    )

    return jnp.asarray(res.x, dtype=DTYPE), res


# ============================================================
# Training
# ============================================================
def train_traditional_alm_burgers():
    t_np, x_np, usol_np, nu = load_burgers_mat(DATA_PATH)

    x_min, x_max = float(x_np.min()), float(x_np.max())
    t_min, t_max = float(t_np.min()), float(t_np.max())

    x_grid = jnp.array(x_np, dtype=DTYPE)
    u0_grid = jnp.array(usol_np[0, :], dtype=DTYPE)

    key = random.PRNGKey(SEED)
    key, k_params, k_obj, k_pde, k_bc, k_ic = random.split(key, 6)

    layer_sizes = [2] + [HIDDEN_DIM] * NUM_HIDDEN + [1]
    params0 = init_mlp_params(k_params, layer_sizes)
    theta, shapes = flatten_params(params0)

    # Fixed points, sampled once.
    X_obj = sample_pde_obj_fixed(k_obj, x_min, x_max, t_min, t_max)
    X_pde_con = sample_pde_con_fixed(k_pde, x_min, x_max, t_min, t_max)
    X_bc_L, X_bc_R = sample_bc_fixed(k_bc, x_min, x_max, t_min, t_max)
    X_ic, u0_ic = sample_ic_fixed(k_ic, x_min, x_max, t_min, x_grid, u0_grid)

    n_pde, n_bc, n_ic, m_con = get_constraint_sizes(USE_PDE_ANCHORS, BC_MODE)

    lam_pde = jnp.zeros((n_pde,), dtype=DTYPE)
    lam_bc = jnp.zeros((n_bc,), dtype=DTYPE)
    lam_ic = jnp.zeros((n_ic,), dtype=DTYPE)

    rho_pde = float(RHO_PDE0)
    rho_bc = float(RHO_BC0)
    rho_ic = float(RHO_IC0)

    prev_pde_rms = None
    prev_bc_rms = None
    prev_ic_rms = None

    best_rel = float("inf")
    best_theta = theta

    print("loaded burgers.mat:", "t", t_np.shape, "x", x_np.shape, "usol", usol_np.shape, "nu", nu)
    print(f"Domain: x in [{x_min}, {x_max}], t in [{t_min}, {t_max}]")
    print("Traditional ALM-PINN with L-BFGS inner solve:")
    print("  fixed points")
    print("  no jitter")
    print("  no resampling")
    print("  theta inner optimizer: scipy L-BFGS-B")
    print("  lambda update: lambda <- lambda + rho*c")
    print("  rho update: blockwise if feasibility stalls")
    print("BC_MODE =", BC_MODE)
    print("USE_PDE_ANCHORS =", USE_PDE_ANCHORS)
    print("n_params =", int(theta.size))
    print("constraints: pde", n_pde, "bc", n_bc, "ic", n_ic, "total", m_con)
    print("objective points =", K_PDE_OBJ)
    print("network =", layer_sizes)
    print("W_PDE_OBJ =", W_PDE_OBJ)
    print("TOTAL_LBFGS_BUDGET =", TOTAL_LBFGS_BUDGET)
    print("MAX_OUTER =", MAX_OUTER, "INNER_MAXITER =", INNER_MAXITER)
    print("rho0: pde", rho_pde, "bc", rho_bc, "ic", rho_ic)

    t0_wall = time.time()

    # warm-up compile once
    _ = alm_loss_and_grad(
        theta, shapes,
        lam_pde, lam_bc, lam_ic,
        DTYPE(rho_pde), DTYPE(rho_bc), DTYPE(rho_ic),
        X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
        DTYPE(nu), DTYPE(W_PDE_OBJ),
        USE_PDE_ANCHORS, BC_MODE,
    )

    for outer in range(1, MAX_OUTER + 1):
        print(f"\n[OUTER {outer}/{MAX_OUTER}] rho_pde={rho_pde:.3e}, rho_bc={rho_bc:.3e}, rho_ic={rho_ic:.3e}")

        theta, res = lbfgs_inner_solve(
            theta, shapes,
            lam_pde, lam_bc, lam_ic,
            rho_pde, rho_bc, rho_ic,
            X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
            DTYPE(nu), DTYPE(W_PDE_OBJ),
            USE_PDE_ANCHORS, BC_MODE,
            maxiter=INNER_MAXITER,
        )

        loss, _, aux = alm_loss_and_grad(
            theta, shapes,
            lam_pde, lam_bc, lam_ic,
            DTYPE(rho_pde), DTYPE(rho_bc), DTYPE(rho_ic),
            X_obj, X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
            DTYPE(nu), DTYPE(W_PDE_OBJ),
            USE_PDE_ANCHORS, BC_MODE,
        )

        c_pde, c_bc, c_ic = constraint_blocks(
            theta, shapes,
            X_pde_con, X_bc_L, X_bc_R, X_ic, u0_ic,
            DTYPE(nu),
            USE_PDE_ANCHORS, BC_MODE,
        )

        pde_rms, pde_max = block_stats(np.asarray(c_pde))
        bc_rms, bc_max = block_stats(np.asarray(c_bc))
        ic_rms, ic_max = block_stats(np.asarray(c_ic))

        # Classical ALM multiplier update.
        if USE_PDE_ANCHORS:
            lam_pde = lam_pde + DTYPE(rho_pde) * c_pde
        lam_bc = lam_bc + DTYPE(rho_bc) * c_bc
        lam_ic = lam_ic + DTYPE(rho_ic) * c_ic

        # Rho update.
        if USE_PDE_ANCHORS and prev_pde_rms is not None and pde_rms > IMPROVE_THRESHOLD * prev_pde_rms:
            rho_pde = min(RHO_GROWTH * rho_pde, RHO_MAX)

        if prev_bc_rms is not None and bc_rms > IMPROVE_THRESHOLD * prev_bc_rms:
            rho_bc = min(RHO_GROWTH * rho_bc, RHO_MAX)

        if prev_ic_rms is not None and ic_rms > IMPROVE_THRESHOLD * prev_ic_rms:
            rho_ic = min(RHO_GROWTH * rho_ic, RHO_MAX)

        prev_pde_rms = pde_rms
        prev_bc_rms = bc_rms
        prev_ic_rms = ic_rms

        mse, rel_l2, _ = eval_full_grid(theta, shapes, x_np, t_np, usol_np)

        print(
            f"[ALM] outer={outer} "
            f"loss={float(loss):.3e} obj={float(aux['obj']):.3e} "
            f"pde=({pde_rms:.3e},{pde_max:.3e}) "
            f"bc=({bc_rms:.3e},{bc_max:.3e}) "
            f"ic=({ic_rms:.3e},{ic_max:.3e}) "
            f"relL2={rel_l2:.3e} "
            f"lbfgs_success={res.success} nit={res.nit} msg='{res.message}'"
        )

        print(
            f"      lam_rms: "
            f"pde={float(jnp.sqrt(jnp.mean(lam_pde**2)+EPS)) if lam_pde.size else 0.0:.3e}, "
            f"bc={float(jnp.sqrt(jnp.mean(lam_bc**2)+EPS)):.3e}, "
            f"ic={float(jnp.sqrt(jnp.mean(lam_ic**2)+EPS)):.3e}"
        )

        if rel_l2 < best_rel:
            best_rel = rel_l2
            best_theta = theta
            np.save(SAVE_NAME, np.asarray(best_theta))
            print(f"      >>> new best relL2={best_rel:.3e}; saved -> {SAVE_NAME}")

    elapsed = time.time() - t0_wall
    print(f"\n[done] elapsed={elapsed:.2f}s")

    mse, rel_l2, _ = eval_full_grid(theta, shapes, x_np, t_np, usol_np)
    print(f"[FINAL] full-grid MSE={mse:.3e}, relL2={rel_l2:.3e}")

    mse_b, rel_b, _ = eval_full_grid(best_theta, shapes, x_np, t_np, usol_np)
    print(f"[BEST]  full-grid MSE={mse_b:.3e}, relL2={rel_b:.3e}")

    np.save(SAVE_NAME, np.asarray(best_theta))
    print("Saved best theta ->", SAVE_NAME)

    return best_theta, shapes, (lam_pde, lam_bc, lam_ic)


def main():
    return train_traditional_alm_burgers()


if __name__ == "__main__":
    main()
